In [0]:
%pip install -q nltk torch torchvision --extra-index-url https://download.pytorch.org/whl/cpu

In [0]:
dbutils.library.restartPython()

In [0]:
import os, torch, torchvision, nltk, sklearn
print("Carpeta de trabajo:", os.getcwd())
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| nltk", nltk.__version__, "| sklearn", sklearn.__version__)

In [0]:
import pruebas_conversaciones
aciertos_v1, total_v1 = pruebas_conversaciones.ejecutar_pruebas()

In [0]:
import pandas as pd
import preprocesamiento as pp

ejemplos = [
    "Hola! Ya atendieron mi ticket #48301? Mi correo es ana.ruiz@empresa.com",
    "No quiero el estatus, quiero los comentarios de la 48213",
    "Necesito una sábana de datos en Excel con las ventas de 2026",
]
display(pd.DataFrame([pp.mostrar_pasos(t) for t in ejemplos]))

In [0]:
import dataset_intenciones as di

ds = pd.DataFrame(di.generar_dataset())
print(f"Ejemplos: {len(ds):,} | Columnas: {list(ds.columns)}")
col_int = "intencion" if "intencion" in ds.columns else ds.columns[1]
print("\nEjemplos por intencion:")
print(ds[col_int].value_counts().sort_index().to_string())
print("\nMuestra aleatoria:")
display(ds.sample(12, random_state=1))

retadora = pd.DataFrame(di.prueba_retadora())
print(f"Prueba retadora escrita a mano: {len(retadora)} frases")
display(retadora.head(10))

In [0]:
import base_conocimiento as bc

bc.cargar_base()
r = bc.resumen()
if r is not None:
    tabla = r.reset_index().rename(columns={"index": "tipo"})
    tabla["total"] = tabla.select_dtypes("number").sum(axis=1)
    display(tabla)

In [0]:
import clasificador_intenciones as ci
metricas_texto = ci.entrenar_y_evaluar()
ablacion = ci.ablacion_preprocesamiento()

In [0]:
import dataset_imagenes as dimg
ruta_imagenes = dimg.generar_dataset()
for split in ("train", "val", "test", "ood"):
    print(split, {c: len(list((ruta_imagenes / split / c).glob("*.png"))) for c in dimg.CLASES})

In [0]:
from PIL import Image, ImageDraw

def mosaico(split, ancho=320, alto=200):
    """Una imagen por clase del split indicado, con su etiqueta."""
    lienzo = Image.new("RGB", (ancho * len(dimg.CLASES), alto + 24), "white")
    d = ImageDraw.Draw(lienzo)
    for i, c in enumerate(dimg.CLASES):
        img = Image.open(sorted((ruta_imagenes / split / c).glob("*.png"))[0]).convert("RGB")
        img.thumbnail((ancho - 8, alto))
        lienzo.paste(img, (i * ancho + 4, 24))
        d.text((i * ancho + 6, 6), c, fill="black")
    return lienzo

for split in ("test", "ood"):
    print(f"Conjunto {split}:")
    display(mosaico(split))

In [0]:
import clasificador_imagenes as cimg
metricas_imagen = cimg.entrenar("efficientnet_b0", epocas_f1=5, epocas_f2=3)

In [0]:
import pruebas_multimodales
resultado = pruebas_multimodales.ejecutar_pruebas()

In [0]:
from PIL import Image
from pathlib import Path

for png in sorted(Path("salidas").glob("matriz_confusion_*.png")):
    print(png.name)
    display(Image.open(png))